# ITI113 - Real-Time Credit Card Fraud Prediction Application

This notebook builds a Gradio application for real-time credit card fraud prediction using the approved SageMaker endpoint. 
It loads the leakage-safe champion’s inference contract, validates transaction inputs, invokes the deployed model and presents the fraud probability and classification result through a user-friendly interface.

1. Install the Application Dependencies
2. Configure the Application and Discover the Endpoint
3. Define Input Validation and Real-Time Fraud Prediction
4. Build and Launch the Gradio Fraud Prediction Application

## 1. Install the Application Dependencies
Run this once in SageMaker Studio. If Gradio is already installed, this cell will complete quickly.

In [1]:
%pip install --quiet "gradio>=5,<7" boto3 pandas jupyter-server-proxy

Note: you may need to restart the kernel to use updated packages.


## 2. Configure the Application and Check the Endpoint
The default endpoint is the Team07 endpoint.

In [2]:
import json
import os
import time
from pathlib import Path

import boto3
import gradio as gr
import pandas as pd
from IPython.display import HTML, display
from botocore.exceptions import BotoCoreError, ClientError

REGION = "ap-southeast-1"
TEAM_ID = "team07"
STUDENT_ID = "s701"
PROJECT_NAME = "credit-card-fraud-detection"
CHAMPION_FILE = Path(f"{TEAM_ID}_best_model.json")
DATASET_PATH = Path("data/fraud_dataset.csv")
FORBIDDEN_FIELDS = {"Risk_Score", "Fraud_Label", "Transaction_ID", "User_ID", "Is_Weekend"}

sts = boto3.client("sts", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

if not CHAMPION_FILE.exists():
    raise FileNotFoundError(f"Missing {CHAMPION_FILE}. Copy the latest Notebook 2 output here.")
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing {DATASET_PATH}. Copy the approved 50K dataset here.")

champion = json.loads(CHAMPION_FILE.read_text(encoding="utf-8"))
RAW_INFERENCE_COLUMNS = champion["raw_inference_columns"]
test_data = pd.read_csv(DATASET_PATH)

missing_columns = [column for column in RAW_INFERENCE_COLUMNS if column not in test_data.columns]
if missing_columns:
    raise RuntimeError(f"Dataset is missing inference fields: {missing_columns}")

response = sm.list_endpoints(SortBy="CreationTime", SortOrder="Descending", MaxResults=100)
team_endpoints = [
    endpoint for endpoint in response["Endpoints"]
    if TEAM_ID in endpoint["EndpointName"].lower()
    and endpoint["EndpointStatus"] == "InService"
]
ENDPOINT_CHOICES = [endpoint["EndpointName"] for endpoint in team_endpoints]
if not ENDPOINT_CHOICES:
    raise RuntimeError(f"No InService endpoint found for {TEAM_ID} in {REGION}.")
DEFAULT_ENDPOINT = next((name for name in ENDPOINT_CHOICES if "v9" in name.lower()), ENDPOINT_CHOICES[0])

def payload_from_row(row_index):
    return json.loads(
        test_data.loc[[row_index], RAW_INFERENCE_COLUMNS].to_json(orient="records", date_format="iso")
    )[0]

normal_index = int(test_data.index[test_data["Fraud_Label"] == 0][0]) if "Fraud_Label" in test_data else 0
fraud_index = int(test_data.index[test_data["Fraud_Label"] == 1][0]) if "Fraud_Label" in test_data else 0
DEFAULT_PAYLOAD_TEXT = json.dumps(payload_from_row(normal_index), indent=2)

print("AWS identity:", sts.get_caller_identity()["Arn"])
print("Endpoint:", DEFAULT_ENDPOINT)
print("Inference fields:", len(RAW_INFERENCE_COLUMNS))
print("Gradio version:", gr.__version__)

AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team07/SageMaker
Endpoint: team07-fraud-v12-endpoint-1787553989
Inference fields: 16
Gradio version: 6.25.0


## 3. Define Input Validation and Real-Time Fraud Prediction
The app validates the selected endpoint and the leakage-safe JSON contract before invoking SageMaker Runtime. Errors are returned inside the interface rather than terminating the notebook.

In [3]:
def endpoint_status(endpoint_name):
    try:
        details = sm.describe_endpoint(EndpointName=endpoint_name)
        status = details["EndpointStatus"]
        icon = "✅" if status == "InService" else "⚠️"
        return (
            f"{icon} **{status}**  \n"
            f"Endpoint: `{endpoint_name}`  \n"
            f"Last modified: `{details['LastModifiedTime']}`"
        )
    except (BotoCoreError, ClientError) as error:
        return f"❌ Unable to read endpoint status: `{error}`"

def load_sample_values(sample_kind):
    row_index = fraud_index if sample_kind == "Fraud example" else normal_index
    payload = payload_from_row(row_index)
    return [payload[column] for column in RAW_INFERENCE_COLUMNS]

def validate_payload(payload):
    supplied_forbidden = sorted(FORBIDDEN_FIELDS & set(payload))
    if supplied_forbidden:
        raise ValueError(f"Forbidden fields supplied: {supplied_forbidden}")
    missing = [column for column in RAW_INFERENCE_COLUMNS if column not in payload]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")
    empty = [column for column, value in payload.items() if value is None or value == ""]
    if empty:
        raise ValueError(f"Complete these fields before prediction: {empty}")
    return payload

def invoke_endpoint(*field_values):
    endpoint_name = DEFAULT_ENDPOINT
    started = time.perf_counter()
    payload = dict(zip(RAW_INFERENCE_COLUMNS, field_values))
    try:
        details = sm.describe_endpoint(EndpointName=endpoint_name)
        if details["EndpointStatus"] != "InService":
            raise RuntimeError(f"Endpoint status is {details['EndpointStatus']}, not InService.")
        payload = validate_payload(payload)
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(payload).encode("utf-8"),
        )
        prediction = json.loads(response["Body"].read().decode("utf-8"))
        first = prediction[0] if isinstance(prediction, list) and prediction else prediction
        label = first.get("label", "Unknown") if isinstance(first, dict) else "Unknown"
        probability = first.get("fraud_probability") if isinstance(first, dict) else None
        threshold = first.get("decision_threshold") if isinstance(first, dict) else None
        probability_text = f"{float(probability):.2%}" if probability is not None else "N/A"
        threshold_text = f"{float(threshold):.4f}" if threshold is not None else "N/A"
        icon = "🚨" if label == "Fraud" else "✅"
        summary = (
            f"## Prediction Result\n\n"
            f"### {icon} {label}\n\n"
            f"- **Fraud probability:** {probability_text}\n"
            f"- **Decision threshold:** {threshold_text}\n\n"
            f"**Decision rule**\n\n"
            f"- Probability ≥ threshold → **Fraud**\n"
            f"- Probability < threshold → **Non-fraud**"
        )
        return summary
    except (ValueError, RuntimeError, BotoCoreError, ClientError) as error:
        message = str(error)
    return f"## Prediction Result\n\n### ❌ Request failed\n\n{message}"

## 4. Build and Launch the Gradio Fraud Prediction Application through the Jupyter proxy

In [4]:
# Close a previous Gradio server when this cell is rerun.
if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass

default_payload = payload_from_row(normal_index)

def make_field_component(column):
    label = column.replace("_", " " )
    value = default_payload[column]
    series = test_data[column]
    if column == "Timestamp":
        return gr.Textbox(
            value=str(value),
            label=label,
            info="Use a parseable timestamp, for example 2026-08-22 14:30:00",
            interactive=True,
        )
    if pd.api.types.is_numeric_dtype(series):
        precision = 0 if pd.api.types.is_integer_dtype(series) else None
        return gr.Number(value=value, label=label, precision=precision, interactive=True)
    choices = sorted(series.dropna().astype(str).unique().tolist())
    if 0 < len(choices) <= 50:
        return gr.Dropdown(choices=choices, value=str(value), label=label, interactive=True)
    return gr.Textbox(value=str(value), label=label, interactive=True)

with gr.Blocks(title="Credit Fraud Prediction App") as demo:
    gr.HTML("<h1 style='text-align:center'>💳 Real-Time Credit Card Fraud Risk Prediction App</h1>")
    gr.Markdown("Using a guided transaction form, predict credit card fraud risk based on various transaction parameters.")

    field_components = []
    with gr.Accordion("Transaction input fields", open=True):
        for start in range(0, len(RAW_INFERENCE_COLUMNS), 3):
            with gr.Row():
                for column in RAW_INFERENCE_COLUMNS[start:start + 3]:
                    component = make_field_component(column)
                    field_components.append(component)

    with gr.Row():
        with gr.Column(scale=2):
            sample_kind = gr.Radio(
                ["Normal example", "Fraud example"],
                value="Normal example",
                label="Load an evaluation example",
            )
            load_button = gr.Button("Load selected example")
            predict_button = gr.Button("Predict fraud risk", variant="primary")

        with gr.Column(scale=2):
            result_summary = gr.Markdown()

    load_button.click(load_sample_values, inputs=sample_kind, outputs=field_components)
    predict_button.click(
        invoke_endpoint,
        inputs=field_components,
        outputs=result_summary,
        api_name="predict_fraud",
    )

GRADIO_PORT = 7860
notebook_base = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")
if not notebook_base:
    # SageMaker's URL segment is the JupyterLab app name, not the space name.
    # Shared spaces normally run a JupyterLab app named 'default'.
    app_name = os.environ.get("SAGEMAKER_APP_NAME", "default")
    notebook_base = f"/jupyterlab/{app_name}/"
if not notebook_base.startswith("/"):
    notebook_base = f"/{notebook_base}"
if not notebook_base.endswith("/"):
    notebook_base += "/"
proxy_path = f"{notebook_base}proxy/{GRADIO_PORT}/"

demo.queue(default_concurrency_limit=2).launch(
    server_name="0.0.0.0",
    server_port=GRADIO_PORT,
    root_path=proxy_path.rstrip("/"),
    share=False,
    inline=False,
    prevent_thread_lock=True,
    show_error=True,
)

display(HTML(
    f"<div style='padding:2px;border:1px solid #22c55e;border-radius:10px'>"
    f"<b>Gradio is running securely through SageMaker Studio.</b><br><br>"
    f"<a href='{proxy_path}' target='_blank' "
    f"style='background:#2563eb;color:white;padding:2px 16px;border-radius:7px;text-decoration:none'>"
    f"Open Gradio endpoint tester</a><br><br>"
    f"<code>{proxy_path}</code></div>"
))

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.
